# K-ABENA — MLP : PyTorch ↔ TensorFlow

Ce notebook présente **le même tutoriel MLP en deux versions** — PyTorch et TensorFlow — avec une structure identique section par section.
Objectif : faciliter la migration d'une plateforme à l'autre.

| Section | PyTorch | TensorFlow |
|---------|---------|------------|
| Import K-ABENA | `kabena_filter_torch(losses, K)` | `KabenaCallback(K=...)` |
| Modèle | `nn.Sequential(...)` | `tf.keras.Sequential(...)` |
| Boucle | `loss.backward()` | `tape.gradient(loss, vars)` |
| Coût adoption | **+2 lignes** | **+1 callback** |

---

## Données communes aux deux versions

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Même dataset pour les deux plateformes
X, y = make_classification(n_samples=2000, n_features=20, n_informative=12, random_state=42)
X    = StandardScaler().fit_transform(X).astype(np.float32)
y    = y.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Hyperparamètres K-ABENA communs
K_ABENA = 0.25
N_ABENA = 0.3
EPOCHS  = 80
LR      = 0.05

---
# VERSION PYTORCH
> **Coût d'adoption : +2 lignes** dans la boucle d'entraînement

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from kabena_ml.integrations.torch_utils import kabena_filter_torch

# ── Modèle PyTorch ────────────────────────────────────────────────────────
class MLP_PT(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def forward(self, x): return self.net(x)

model_pt  = MLP_PT()
opt_pt    = torch.optim.SGD(model_pt.parameters(), lr=LR, momentum=0.9)
X_pt, y_pt = torch.tensor(X_train), torch.tensor(y_train)
history_pt = []

# ── Boucle PyTorch + K-ABENA ──────────────────────────────────────────────
for epoch in range(EPOCHS):
    # AVANT K-ABENA : loss = F.cross_entropy(model_pt(X_pt), y_pt)
    losses = F.cross_entropy(model_pt(X_pt), y_pt, reduction='none')  # ← 'none'
    mask   = kabena_filter_torch(losses, K=K_ABENA, N=N_ABENA)        # ← +1 ligne
    L_KA   = losses[mask].mean()

    opt_pt.zero_grad(); L_KA.backward(); opt_pt.step()
    m = mask.sum().item()
    history_pt.append({"epoch": epoch, "loss": L_KA.item(),
                        "m": m, "gain_pct": round((1 - m/len(y_pt))*100)})

# ── Évaluation PyTorch ────────────────────────────────────────────────────
with torch.no_grad():
    X_te_pt  = torch.tensor(X_test)
    preds_pt = model_pt(X_te_pt).argmax(1).numpy()
    acc_pt   = (preds_pt == y_test).mean()

print(f"[PyTorch]  Accuracy : {acc_pt:.4f} | Gain moyen : {np.mean([r['gain_pct'] for r in history_pt]):.1f}%")

---
# VERSION TENSORFLOW
> **Coût d'adoption : +1 callback** dans model.fit()

In [ ]:
import tensorflow as tf
from kabena_ml.integrations.tf_utils import KabenaCallback, KabenaTFTrainer
from kabena_ml import KabenaConfig

# ── Modèle TensorFlow (même architecture que PyTorch) ─────────────────────
model_tf = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(20,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(2),
])

# ── Option A : Keras callback (+1 argument dans model.fit) ────────────────
model_tf.compile(
    optimizer=tf.keras.optimizers.SGD(LR, momentum=0.9),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

ka_cb = KabenaCallback(K=K_ABENA, N=N_ABENA, verbose=False)

# AVANT K-ABENA : model_tf.fit(X_train, y_train, epochs=EPOCHS, verbose=0)
model_tf.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=len(X_train),  # batch complet = même que PyTorch ci-dessus
    verbose=0,
    callbacks=[ka_cb]   # ← +1 ligne
)

_, acc_tf = model_tf.evaluate(X_test, y_test, verbose=0)
print(f"[TensorFlow Callback]  Accuracy : {acc_tf:.4f} | {ka_cb.stats_summary()}")

# ── Option B : GradientTape (équivalent exact de la boucle PyTorch) ───────
model_tf2 = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(20,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(2),
])
cfg     = KabenaConfig(K=K_ABENA, N=N_ABENA, task='classification')
trainer = KabenaTFTrainer(model_tf2, cfg, tf.keras.optimizers.SGD(LR, momentum=0.9))
trainer.fit(tf.constant(X_train), tf.constant(y_train), epochs=EPOCHS, batch_size=len(X_train))

logits2 = model_tf2(X_test, training=False)
acc_tf2 = (tf.argmax(logits2, axis=1).numpy() == y_test).mean()
print(f"[TensorFlow GradTape]  Accuracy : {acc_tf2:.4f}")

---
## Tableau comparatif des résultats

In [ ]:
import pandas as pd
df = pd.DataFrame({
    'Plateforme': ['PyTorch', 'TF Callback', 'TF GradientTape'],
    'Accuracy':   [acc_pt,    acc_tf,         acc_tf2],
    'K':          [K_ABENA]*3,
    'N':          [N_ABENA]*3,
    'Coût adoption': ['+2 lignes', '+1 callback', '+2 lignes'],
})
print(df.to_string(index=False))

## Guide de migration PyTorch → TensorFlow

| Concept | PyTorch | TensorFlow |
|---------|---------|------------|
| Import filtre | `from kabena_ml.integrations.torch_utils import kabena_filter_torch` | `from kabena_ml.integrations.tf_utils import KabenaCallback` |
| Loss individuelle | `F.cross_entropy(..., reduction='none')` | automatique dans callback |
| Appliquer le filtre | `mask = kabena_filter_torch(losses, K, N)` | `callbacks=[KabenaCallback(K, N)]` |
| Loss filtrée | `losses[mask].mean().backward()` | automatique |
| GradientTape | `losses[mask].mean()` → `tape.gradient(...)` | `tf.boolean_mask(losses, mask)` |

**Règle d'or** : la logique K-ABENA est identique dans les deux frameworks —
seule la syntaxe de manipulation des tenseurs change.